<a href="https://colab.research.google.com/github/ibtisamelahi07/generativeai/blob/main/medical_finetuning_qlora_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2: Medical Fine-tuning With QLoRA and Unsloth

This notebook implements a complete QLoRA-based fine-tuning workflow for medical language models using Unsloth's optimized framework.

## What This Notebook Does:
1. Sets up Unsloth for efficient 4-bit quantized training
2. Loads a medical Q&A dataset
3. Configures LoRA adapters for parameter-efficient fine-tuning
4. Trains the model with memory monitoring
5. Tests on medical queries
6. Saves the fine-tuned adapter

## Requirements:
- Google Colab with GPU (T4 or better recommended)
- ~15GB GPU memory for 7B models
- Runtime: GPU (Settings → Change runtime type → T4 GPU)

## Step 1: Install Unsloth and Dependencies

In [ ]:
%%capture
# Install Unsloth - optimized for QLoRA training
!pip install unsloth
# Also install xformers for memory efficiency
!pip install --no-deps xformers

## Step 2: Import Libraries and Setup

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import gc

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.64 GB


## Step 3: Configure Model Parameters

In [ ]:
# Model configuration
max_seq_length = 2048  # Maximum sequence length
dtype = None  # Auto-detect dtype
load_in_4bit = True  # Use 4-bit quantization for memory efficiency

# Choose your base model (uncomment one)
model_name = "unsloth/llama-3-8b-bnb-4bit"  # Llama 3 8B (recommended)
# model_name = "unsloth/mistral-7b-bnb-4bit"  # Alternative: Mistral 7B
# model_name = "unsloth/Phi-3-mini-4k-instruct"  # Alternative: Phi-3

print(f"Loading model: {model_name}")
print(f"4-bit quantization: {load_in_4bit}")
print(f"Max sequence length: {max_seq_length}")

Loading model: unsloth/llama-3-8b-bnb-4bit
4-bit quantization: True
Max sequence length: 2048


## Step 4: Load Model with Unsloth

In [ ]:
# Load model with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Model loaded successfully!")
print(f"Model type: {type(model)}")

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Model loaded successfully!
Model type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>


## Step 5: Configure LoRA Adapters

In [ ]:
# LoRA configuration for parameter-efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA rank - higher = more parameters but better accuracy
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],  # Which layers to adapt
    lora_alpha=16,  # LoRA scaling parameter
    lora_dropout=0,  # Dropout for LoRA layers (0 = no dropout)
    bias="none",  # Don't train bias parameters
    use_gradient_checkpointing="unsloth",  # Memory-efficient training
    random_state=3407,
    use_rslora=False,  # Rank-stabilized LoRA
    loftq_config=None,  # LoftQ initialization
)

print("LoRA adapters configured!")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Unsloth 2026.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapters configured!
Trainable parameters: 41,943,040
Total parameters: 4,582,543,360


## Step 6: Load Medical Dataset

 use a medical Q&A dataset.

In [ ]:
# Option 1: Load from Hugging Face (medical Q&A dataset)
# This uses a sample medical dataset - replace with your preferred medical dataset
try:
    # Try loading a medical QA dataset
    dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")
    print(f"Loaded medical flashcard dataset: {len(dataset)} examples")
except:
    try:
        # Fallback to another medical dataset
        dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train[:5000]")  # Use subset
        print(f"Loaded ChatDoctor dataset: {len(dataset)} examples")
    except:
        # If datasets fail, create a small sample dataset
        print("Creating sample medical dataset...")
        sample_data = {
            "instruction": [
                "What are the symptoms of diabetes?",
                "How is hypertension treated?",
                "What causes asthma attacks?",
                "Explain the difference between Type 1 and Type 2 diabetes.",
                "What are the risk factors for heart disease?"
            ],
            "output": [
                "Common symptoms of diabetes include increased thirst, frequent urination, extreme hunger, unexplained weight loss, fatigue, blurred vision, and slow-healing sores.",
                "Hypertension is typically treated through lifestyle modifications (diet, exercise, weight loss) and medications such as ACE inhibitors, ARBs, beta-blockers, or diuretics.",
                "Asthma attacks are triggered by allergens (pollen, dust, pet dander), respiratory infections, exercise, cold air, air pollution, or strong emotions.",
                "Type 1 diabetes is an autoimmune condition where the pancreas produces little or no insulin. Type 2 diabetes is characterized by insulin resistance where the body doesn't use insulin effectively.",
                "Risk factors for heart disease include high blood pressure, high cholesterol, smoking, diabetes, obesity, physical inactivity, family history, and age."
            ]
        }
        from datasets import Dataset
        dataset = Dataset.from_dict(sample_data)

# Display sample
print("\nSample data:")
print(dataset[0])

README.md:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

Loaded medical flashcard dataset: 33955 examples

Sample data:
{'input': 'What is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?', 'output': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.', 'instruction': 'Answer this question truthfully'}


## Step 7: Prepare Training Data with Prompts

In [ ]:
# Create a prompt template for medical Q&A
alpaca_prompt = """Below is an instruction that describes a medical question. Write a response that appropriately answers the question.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # End of sequence token

def formatting_prompts_func(examples):
    """Format examples into training prompts"""
    instructions = examples.get("instruction") or examples.get("input") or examples.get("question")
    outputs = examples.get("output") or examples.get("response") or examples.get("answer")

    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print("Dataset formatted!")
print(f"\nExample formatted prompt:\n{dataset[0]['text'][:500]}...")

Map:   0%|          | 0/33955 [00:00<?, ? examples/s]

Dataset formatted!

Example formatted prompt:
Below is an instruction that describes a medical question. Write a response that appropriately answers the question.

### Instruction:
Answer this question truthfully

### Response:
Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.<|end_of_text|>...


## Step 8: Configure Training Parameters

In [ ]:
# Training configuration
training_args = TrainingArguments(
    # Output and logging
    output_dir="./medical_model_output",
    logging_steps=10,
    logging_dir="./logs",

    # Training parameters
    per_device_train_batch_size=2,  # Batch size per GPU
    gradient_accumulation_steps=4,  # Effective batch size = 2 * 4 = 8
    warmup_steps=5,  # Learning rate warmup
    num_train_epochs=3,  # Number of training epochs (adjust as needed)
    # max_steps=60,  # Alternative: set max steps instead of epochs

    # Optimization
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),  # Use FP16 if BF16 not available
    bf16=torch.cuda.is_bf16_supported(),  # Use BF16 if available
    optim="adamw_8bit",  # 8-bit optimizer for memory efficiency
    weight_decay=0.01,
    lr_scheduler_type="linear",

    # Checkpointing
    save_strategy="epoch",
    save_total_limit=2,  # Keep only last 2 checkpoints

    # Other settings
    seed=3407,
    report_to="none",  # Disable wandb/tensorboard
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training configuration:
  Epochs: 3
  Batch size: 2
  Gradient accumulation: 4
  Effective batch size: 8
  Learning rate: 0.0002


## Step 9: Initialize Trainer and Start Training

In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can set to True for efficiency if sequences are short
    args=training_args,
)

print("Trainer initialized!")
print("\n" + "="*50)
print("STARTING TRAINING")
print("="*50)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/33955 [00:00<?, ? examples/s]

Trainer initialized!

STARTING TRAINING


## Step 10: Monitor Memory and Train

In [ ]:
# Check memory before training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"GPU Memory before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# Train the model
trainer_stats = trainer.train()

# Check memory after training
if torch.cuda.is_available():
    print(f"\nGPU Memory after training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"Peak GPU Memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

print("\n" + "="*50)
print("TRAINING COMPLETED!")
print("="*50)
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training loss: {trainer_stats.metrics['train_loss']:.4f}")

GPU Memory before training: 5.89 GB
GPU Memory reserved: 5.90 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 33,955 | Num Epochs = 3 | Total steps = 12,735
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.616230
20,0.833887
30,0.795333
40,0.827174
50,0.808137
60,0.824625
70,0.746703
80,0.742302
90,0.739314
100,0.767114


KeyboardInterrupt: 

In [ ]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test medical queries
test_questions = [
    "What are the early signs of Alzheimer's disease?",
    "How can I lower my cholesterol naturally?",
    "What is the difference between a virus and bacteria?",
    "What are the symptoms of a heart attack?",
]

print("\n" + "="*50)
print("TESTING FINE-TUNED MODEL")
print("="*50 + "\n")

for question in test_questions:
    # Format the question
    prompt = alpaca_prompt.format(question, "")

    # Tokenize
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        use_cache=True,
    )

    # Decode and display
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the response part
    response = response.split("### Response:")[-1].strip()

    print(f"Q: {question}")
    print(f"A: {response}")
    print("\n" + "-"*50 + "\n")

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



TESTING FINE-TUNED MODEL



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Q: What are the early signs of Alzheimer's disease?
A: The early signs of Alzheimer's disease include changes in mood and personality, such as depression, anxiety, and irritability.

--------------------------------------------------



Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How can I lower my cholesterol naturally?
A: I can lower my cholesterol naturally by eating a healthy diet that is low in saturated fat and cholesterol, and by exercising regularly.

--------------------------------------------------



Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the difference between a virus and bacteria?
A: Viruses are made up of nucleic acid surrounded by a protein coat, while bacteria are made up of a cell wall, cell membrane, cytoplasm, and nucleic acid.

--------------------------------------------------

Q: What are the symptoms of a heart attack?
A: The symptoms of a heart attack include chest pain, which is described as a crushing or squeezing sensation in the chest that may radiate to the left arm, jaw, neck, or back; shortness of breath; and sweating.

--------------------------------------------------



## Step 12: Save the Fine-tuned Adapter

In [ ]:
# Save LoRA adapter
model.save_pretrained("medical_lora_adapter")
tokenizer.save_pretrained("medical_lora_adapter")

print("Model adapter saved to 'medical_lora_adapter' directory!")
print("\nYou can load this adapter later with:")
print("model, tokenizer = FastLanguageModel.from_pretrained('medical_lora_adapter')")

Unsloth: Restored added_tokens_decoder metadata in medical_lora_adapter/tokenizer_config.json.


Model adapter saved to 'medical_lora_adapter' directory!

You can load this adapter later with:
model, tokenizer = FastLanguageModel.from_pretrained('medical_lora_adapter')


## Step 13:Save to Hugging Face Hub

In [ ]:
# Uncomment to push to Hugging Face Hub
# You'll need to login with: huggingface-cli login

# model.push_to_hub("your-username/medical-llama3-qlora", token="your_token_here")
# tokenizer.push_to_hub("your-username/medical-llama3-qlora", token="your_token_here")

print("To upload to Hugging Face:")
print("1. Login: !huggingface-cli login")
print("2. Uncomment and run the push_to_hub commands above")

To upload to Hugging Face:
1. Login: !huggingface-cli login
2. Uncomment and run the push_to_hub commands above


## Step 14: Merge and Save Full Model

In [ ]:
# Optional: Merge LoRA weights into base model for standalone deployment
# WARNING: This requires more memory

# model.save_pretrained_merged(
#     "medical_model_merged",
#     tokenizer,
#     save_method="merged_16bit",  # or "merged_4bit" for quantized version
# )

print("To save merged model:")
print("Uncomment the code above (requires more memory)")
print("This creates a standalone model without needing the base model + adapter")

To save merged model:
Uncomment the code above (requires more memory)
This creates a standalone model without needing the base model + adapter


In [ ]:
# 1. Model ko "Testing Mode" mein layein (Fast inference)
FastLanguageModel.for_inference(model)

# 2. Apna medical test question likhein
test_question = "A 45-year-old patient comes in with frequent urination, excessive thirst, and unexplained weight loss. What is the most likely diagnosis?"

# 3. Question ko model ki language mein format karein
inputs = tokenizer(
[
    alpaca_prompt.format(
        test_question, # Hamara sawal
        "", # <think> wala hissa khali chhor dein, model khud soche ga
        ""  # Response bhi khali chhor dein
    )
], return_tensors = "pt").to("cuda")

# 4. Model se answer generate karwayein
print("Doctor AI is thinking...\n")
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 800, # Model ko lamba answer likhne ki space dena
    use_cache = True,
)

# 5. Final output print karein
print(tokenizer.batch_decode(outputs)[0])

Both `max_new_tokens` (=800) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Doctor AI is thinking...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


<|begin_of_text|>Below is an instruction that describes a medical question. Write a response that appropriately answers the question.

### Instruction:
A 45-year-old patient comes in with frequent urination, excessive thirst, and unexplained weight loss. What is the most likely diagnosis?

### Response:
The most likely diagnosis for this patient is diabetes mellitus, which is a metabolic disorder characterized by high blood sugar levels. Diabetes is caused by a lack of insulin production or insulin resistance, leading to an accumulation of glucose in the bloodstream. The symptoms of diabetes can vary depending on the severity of the condition, but they typically include frequent urination, excessive thirst, and unexplained weight loss. Treatment for diabetes may involve lifestyle changes, such as diet and exercise, as well as medication to manage blood sugar levels. It is important to consult with a healthcare provider for an accurate diagnosis and appropriate treatment plan.<|end_of_t

## Summary and Next Steps

### What You've Accomplished:
✅ Loaded a base model with 4-bit quantization  
✅ Configured LoRA adapters for efficient fine-tuning  
✅ Trained on medical Q&A data  
✅ Monitored memory usage throughout training  
✅ Tested the model on new medical queries  
✅ Saved the fine-tuned adapter  

### Memory Efficiency:
- **4-bit quantization**: Reduces model size by ~75%
- **LoRA adapters**: Only trains <1% of parameters
- **Gradient checkpointing**: Trades computation for memory
- **Result**: 7-8B models trainable on free Colab T4 GPU!

### To Improve Your Model:
1. **Use more data**: Train on larger medical datasets
2. **Train longer**: Increase epochs or max_steps
3. **Tune hyperparameters**: Adjust learning rate, LoRA rank, batch size
4. **Use better base models**: Try larger models if you have more GPU memory
5. **Domain-specific data**: Use clinical notes, medical textbooks, or specialized Q&A

### Loading Your Model Later:
```python
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="medical_lora_adapter",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

### Resources:
- Unsloth Documentation: https://github.com/unslothai/unsloth
- Medical Datasets: https://huggingface.co/datasets?search=medical
- LoRA Paper: https://arxiv.org/abs/2106.09685